In [ ]:
import os
import sys
from pathlib import Path

import pandas as pd
import numpy as np
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import (accuracy_score, roc_auc_score, average_precision_score,
                             precision_recall_curve, confusion_matrix,
                             precision_recall_fscore_support, roc_curve, auc,
                             classification_report
                             )
from sklearn.feature_selection import SelectKBest, f_classif
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import joblib

# Ensure the project root is on Python's import path so notebook imports work.
notebook_dir = Path().resolve()
project_root = notebook_dir.parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from Cleaning.cleandata import load_data

# Load cleaned model data
data_path = project_root / "Cleaning" / "FRED_MD_Cleaned.csv"
df = pd.read_csv(data_path, parse_dates=["sasdate"])
X, y = load_data(data_path)


In [ ]:
# Load tuned models and prepare visualization variables
xgb_model_path = project_root / "Models" / "outputs" / "xgb_best.joblib"
rf_model_path = project_root / "Models" / "outputs" / "rf_best.joblib"

if not xgb_model_path.exists():
    raise FileNotFoundError(f"Tuned XGBoost model not found at {xgb_model_path}. Run model_tuning.py first.")
if not rf_model_path.exists():
    raise FileNotFoundError(f"Tuned Random Forest model not found at {rf_model_path}. Run model_tuning.py first.")

xgb_model = joblib.load(xgb_model_path)
rf_model = joblib.load(rf_model_path)

results_list = []
all_importances = []

# Prepare one fold for detailed ROC/confusion matrix visualization
fold_idx = 2  # choose a specific fold for display
folds = list(TimeSeriesSplit(n_splits=3).split(X))
train_idx, test_idx = folds[fold_idx]
X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

vis_results = {}
for name, model in [("XGBoost", xgb_model), ("RandomForest", rf_model)]:
    y_proba = model.predict_proba(X_test)[:, 1]
    y_pred = (y_proba >= 0.10).astype(int)

    if len(np.unique(y_test)) > 1:
        pr_auc = average_precision_score(y_test, y_proba)
        roc_auc = roc_auc_score(y_test, y_proba)
    else:
        pr_auc, roc_auc = np.nan, np.nan

    p, r, f, s = precision_recall_fscore_support(
        y_test,
        y_pred,
        labels=[1],
        zero_division=0
    )

    metrics = {
        "Accuracy": accuracy_score(y_test, y_pred),
        "PR-AUC": pr_auc,
        "ROC-AUC": roc_auc,
        "Precision_Class1": p[0],
        "Recall_Class1": r[0],
        "Fold": fold_idx,
        "Model": name
    }

    results_list.append(metrics)
    all_importances.append(model.feature_importances_ if hasattr(model, "feature_importances_") else None)
    vis_results[name] = {
        "model": model,
        "y_proba": y_proba,
        "y_pred": y_pred,
    }

# Keep the existing default variables pointing to XGBoost for earlier notebook cells
y_proba = vis_results["XGBoost"]["y_proba"]
y_pred = vis_results["XGBoost"]["y_pred"]


XGBClassifier | Fold 1 complete | ROC-AUC: 0.8375
XGBClassifier | Fold 2 complete | ROC-AUC: 0.8876
XGBClassifier | Fold 3 complete | ROC-AUC: 0.9501
RandomForestClassifier | Fold 1 complete | ROC-AUC: 0.8142
RandomForestClassifier | Fold 2 complete | ROC-AUC: 0.8783
RandomForestClassifier | Fold 3 complete | ROC-AUC: 0.9325


In [ ]:
# Tuned models are loaded from disk in the previous cell.
# This notebook performs inference and visualization only, without re-training models.


Below, we create visualizations of out models performace and evaluate it by --- different metrics

In [65]:
# Model performance plots using Plotly

# Note: Use y_test and y_proba from your selected fold
precision, recall, _ = precision_recall_curve(y_test, y_proba)
avg_p = average_precision_score(y_test, y_proba)

# ROC and PR curves
fpr, tpr, _ = roc_curve(y_test, y_proba)
roc_auc = auc(fpr, tpr)

fig = make_subplots(rows=1, cols=2, subplot_titles=('ROC Curve', 'Precision-Recall Curve'))
fig.add_trace(go.Scatter(x=fpr, y=tpr, mode='lines', fill='tozeroy', name=f'ROC (AUC={roc_auc:.4f})'), row=1, col=1)
fig.add_trace(go.Scatter(x=[0, 1], y=[0, 1], mode='lines', name='Random Chance', line=dict(dash='dash')), row=1, col=1)
fig.add_trace(go.Scatter(x=recall, y=precision, mode='lines', fill='tozeroy', name=f'PR (AP={avg_p:.4f})'), row=1, col=2)

fig.update_layout(
    title='Single Model Performance: ROC and Precision-Recall',
    height=600,
    legend=dict(x=0.5, y=-0.15, orientation='h')
)
fig.update_xaxes(title_text='False Positive Rate', row=1, col=1)
fig.update_yaxes(title_text='True Positive Rate', row=1, col=1)
fig.update_xaxes(title_text='Recall', row=1, col=2)
fig.update_yaxes(title_text='Precision', row=1, col=2)

fig.show()


In [66]:
# Compare XGBoost and RandomForest with Plotly

roc_fig = go.Figure()
pr_fig = go.Figure()
for name, result in vis_results.items():
    fpr, tpr, _ = roc_curve(y_test, result["y_proba"])
    roc_auc = auc(fpr, tpr)
    roc_fig.add_trace(go.Scatter(x=fpr, y=tpr, mode='lines', name=f"{name} ROC (AUC={roc_auc:.4f})"))

    precision, recall, _ = precision_recall_curve(y_test, result["y_proba"])
    avg_prec = average_precision_score(y_test, result["y_proba"])
    pr_fig.add_trace(go.Scatter(x=recall, y=precision, mode='lines', name=f"{name} PR (AP={avg_prec:.4f})"))

roc_fig.add_trace(go.Scatter(x=[0, 1], y=[0, 1], mode='lines', name='Random Chance', line=dict(dash='dash', color='gray')))
roc_fig.update_layout(
    title='ROC Curves: XGBoost vs RandomForest',
    xaxis_title='False Positive Rate (FPR)',
    yaxis_title='True Positive Rate (TPR)',
    height=600
)
roc_fig.show()

pr_fig.update_layout(
    title='Precision-Recall Curves: XGBoost vs RandomForest',
    xaxis_title='Recall',
    yaxis_title='Precision',
    height=600
)
pr_fig.show()

cm_fig = make_subplots(rows=1, cols=2, subplot_titles=['XGBoost Confusion Matrix', 'RandomForest Confusion Matrix'], shared_yaxes=True)
for idx, (name, result) in enumerate(vis_results.items(), start=1):
    cm = confusion_matrix(y_test, result["y_pred"])
    cm_fig.add_trace(
        go.Heatmap(
            z=cm,
            x=['No Inversion', 'Inversion'],
            y=['No Inversion', 'Inversion'],
            text=cm,
            texttemplate='%{text}',
            colorscale='Blues',
            showscale=False
        ),
        row=1,
        col=idx
    )

cm_fig.update_layout(
    title='Confusion Matrices: XGBoost vs RandomForest',
    height=500
)
cm_fig.show()


In [67]:
# CONFUSION MATRIX

cm = confusion_matrix(y_test, y_pred)
fig = go.Figure(
    data=go.Heatmap(
        z=cm,
        x=['No Inversion', 'Inversion'],
        y=['No Inversion', 'Inversion'],
        text=cm,
        texttemplate='%{text}',
        colorscale='Blues',
        showscale=True
    )
)
fig.update_layout(
    title='Confusion Matrix: Predictive Performance (Fold 3)',
    xaxis_title='Predicted label',
    yaxis_title='True label',
    height=500
)
fig.show()


In [68]:
# FEATURE IMPORTANCE

import plotly.express as px
import plotly.graph_objects as go

# Use the trained models from the selected fold for importance visualization
xgb_importances = vis_results["XGBoost"]["model"].feature_importances_
rf_importances = vis_results["RandomForest"]["model"].feature_importances_

feature_names = X.columns

xgb_df = pd.DataFrame({
    'Feature': feature_names,
    'Importance': xgb_importances
}).sort_values(by='Importance', ascending=False)

rf_df = pd.DataFrame({
    'Feature': feature_names,
    'Importance': rf_importances
}).sort_values(by='Importance', ascending=False)

# Show top 15 features for each model
xgb_top = xgb_df.head(15)[::-1]
rf_top = rf_df.head(15)[::-1]

fig_xgb = px.bar(
    xgb_top,
    x='Importance',
    y='Feature',
    orientation='h',
    title='XGBoost Feature Importance (Top 15)',
    labels={'Importance': 'Importance', 'Feature': 'Feature'}
)
fig_xgb.update_layout(yaxis={'categoryorder':'total ascending'}, height=700)
fig_xgb.show()

fig_rf = px.bar(
    rf_top,
    x='Importance',
    y='Feature',
    orientation='h',
    title='Random Forest Feature Importance (Top 15)',
    labels={'Importance': 'Importance', 'Feature': 'Feature'}
)
fig_rf.update_layout(yaxis={'categoryorder':'total ascending'}, height=700)
fig_rf.show()

combined_df = pd.DataFrame({
    'Feature': feature_names,
    'XGBoost Importance': xgb_importances,
    'RandomForest Importance': rf_importances
}).sort_values(by='XGBoost Importance', ascending=False)

combined_top = combined_df.head(15)
fig_combined = go.Figure()
fig_combined.add_trace(go.Bar(
    x=combined_top['Feature'],
    y=combined_top['XGBoost Importance'],
    name='XGBoost'
))
fig_combined.add_trace(go.Bar(
    x=combined_top['Feature'],
    y=combined_top['RandomForest Importance'],
    name='RandomForest'
))
fig_combined.update_layout(
    title='Top 15 Features: XGBoost vs Random Forest Importance',
    xaxis_title='Feature',
    yaxis_title='Importance',
    barmode='group',
    xaxis_tickangle=-45,
    height=750
)
fig_combined.show()

feat_df = combined_df.head(15)

In [69]:
## Performace Summary 

results_df = pd.DataFrame(results_list)
summary_table = pd.DataFrame({
    'Average': results_df.mean(numeric_only=True),
    'Std_Dev': results_df.std(numeric_only=True)
}).drop('Fold', axis=0)

print("\n" + "="*40)
print("FINAL CROSS-VALIDATION SUMMARY")
print("="*40)
print(results_df.round(4).to_string(index=False))
print("\nAGGREGATED PERFORMANCE:")
print(summary_table.round(4))


FINAL CROSS-VALIDATION SUMMARY
 Accuracy  PR-AUC  ROC-AUC  Precision_Class1  Recall_Class1  Fold  Features                  Model
   0.7432  0.6555   0.8375            0.3175         0.8333     1       123          XGBClassifier
   0.7869  0.3287   0.8876            0.2340         0.7857     2       123          XGBClassifier
   0.9235  0.8317   0.9501            0.8077         0.7000     3       123          XGBClassifier
   0.5902  0.6030   0.8142            0.2198         0.8333     1       123 RandomForestClassifier
   0.7377  0.4645   0.8783            0.2069         0.8571     2       123 RandomForestClassifier
   0.9235  0.8296   0.9325            0.7857         0.7333     3       123 RandomForestClassifier

AGGREGATED PERFORMANCE:
                   Average  Std_Dev
Accuracy            0.7842   0.1267
PR-AUC              0.6188   0.1996
ROC-AUC             0.8834   0.0525
Precision_Class1    0.4286   0.2878
Recall_Class1       0.7905   0.0626
Features          123.0000   0.000